In [1]:
import os
import sys

# --- COLAB SETUP ---
try:
    from google.colab import drive
    IN_COLAB = True
    print("Running in Google Colab")

    # Mount Drive
    drive.mount('/content/drive')

    # Clone Repository (User's Fork)
    REPO_URL = "https://github.com/zeynepoztunc/aml-procedural-mistake-detection.git"
    BRANCH = "emre-second-step" # Branch to checkout

    if not os.path.exists('/content/code'):
        print(f"Cloning repository from {REPO_URL}...")
        !git clone --recursive {REPO_URL} /content/code

        # Checkout Branch
        print(f"Checking out branch: {BRANCH}")
        os.chdir('/content/code')
        !git fetch origin {BRANCH}
        !git checkout {BRANCH}
    else:
        print("Repository already exists.")
        os.chdir('/content/code')
        print(f"Checking out branch: {BRANCH}")
        !git fetch origin {BRANCH}
        !git checkout {BRANCH}
        !git pull origin {BRANCH}

    # Change Directory
    os.chdir('/content/code')
    print(f"Current working directory: {os.getcwd()}")

    # Add to sys.path
    if '/content/code' not in sys.path:
        sys.path.append('/content/code')

    # --- INJECT MODIFIED FILES (EgoVLP Support & Fixes) ---
    # This ensures the Colab environment has the latest changes even if not pushed to GitHub

    print("Injecting modified files...")

    # 1. constants.py
    with open('constants.py', 'w') as f:
        f.write('''class Constants:
    RESULTS = "results"
    RESULT_ID = "result_id"
    DATE = "date"
    RESULT_DETAILS = "result_details"
    BEST_MODEL_TYPE = "best_model_type"

    ACCURACY = "accuracy"
    LOSS = "loss"
    PRECISION = "precision"
    RECALL = "recall"
    F1 = "f1"
    AUC = "auc"
    PR_AUC = "pr_auc"

    STEP_METRICS = "step_metrics"
    SUB_STEP_METRICS = "sub_step_metrics"

    TASK_NAME = "task_name"
    VARIANT = "variant"
    MODEL_NAME = "model_name"
    BACKBONE = "backbone"
    MODALITY = "modality"
    SPLIT = "split"

    TRAIN = "train"
    VAL = "val"
    TEST = "test"

    ERROR_RECOGNITION = "error_recognition"
    ERROR_CATEGORY_RECOGNITION = "error_category_recognition"
    EARLY_ERROR_RECOGNITION = "early_error_recognition"

    RECORDINGS_SPLIT = "recordings"
    PERSON_SPLIT = "person"
    ENVIRONMENT_SPLIT = "environment"
    STEP_SPLIT = "step"

    # --------------------- MODEL SPECIFIC CONSTANTS ---------------------
    OMNIVORE = "omnivore"
    RESNET3D = "3dresnet"
    X3D = "x3d"
    SLOWFAST = "slowfast"
    IMAGEBIND = "imagebind"
    EGOVLP = "egovlp"

    MLP_VARIANT = "MLP"
    TRANSFORMER_VARIANT = "Transformer"
    MULTIMODAL_VARIANT = "Multimodal"

    # ----------------------- WANDB CONSTANTS -----------------------
    WANDB_PROJECT = "error_recognition"

    AUDIO = "audio"
    VIDEO = "video"
    TEXT = "text"
    DEPTH = "depth"

    # ---------------------- ERROR CATEGORY TYPES ----------------------
    TECHNIQUE_ERROR = "Technique Error"
    PREPARATION_ERROR = "Preparation Error"
    TEMPERATURE_ERROR = "Temperature Error"
    MEASUREMENT_ERROR = "Measurement Error"
    TIMING_ERROR = "Timing Error"
''')

    # 2. core/config.py
    with open('core/config.py', 'w') as f:
        f.write('''from argparse import ArgumentParser
from constants import Constants as const


class Config(object):
    """Wrapper class for model hyperparameters."""

    def __init__(self):
        """
        Defaults
        """
        self.backbone = "egovlp"
        self.modality = "video"
        self.phase = "train"
        self.segment_length = 1

        # Use this for 1 sec video features
        self.segment_features_directory = "data/"

        self.ckpt_directory = "/data/rohith/captain_cook/checkpoints/"
        self.split = "recordings"
        self.batch_size = 1
        self.test_batch_size = 1
        self.num_epochs = 10
        self.lr = 1e-3
        self.weight_decay = 1e-3
        self.log_interval = 5
        self.dry_run = False
        self.ckpt = None
        self.seed = 1000
        self.device = "cuda"

        self.variant = const.TRANSFORMER_VARIANT
        self.model_name = None
        self.task_name = const.ERROR_RECOGNITION
        self.error_category = None

        self.enable_wandb = True

        self.parser = self.setup_parser()
        self.args = vars(self.parser.parse_args())
        self.save_model = True
        self.__dict__.update(self.args)

    def setup_parser(self):
        """
        Sets up an argument parser
        :return:
        """
        parser = ArgumentParser(description="training code")

        # ----------------------------------------------------------------------------------------------
        # CONFIGURATION PARAMETERS
        # ----------------------------------------------------------------------------------------------

        parser.add_argument("--batch_size", type=int, default=1, help="batch size")
        parser.add_argument("--test-batch-size", type=int, default=1, help="input batch size for testing (default: 1000)")
        parser.add_argument("--num_epochs", type=int, default=10, help="number of epochs")
        parser.add_argument("--lr", type=float, default=1e-3, help="learning rate")
        parser.add_argument("--weight_decay", type=float, default=1e-3, help="weight decay")
        parser.add_argument("--ckpt", type=str, default=None, help="checkpoint path")
        parser.add_argument("--seed", type=int, default=42, help="random seed (default: 1000)")

        parser.add_argument("--backbone", type=str, default=const.EGOVLP, help="backbone model")
        parser.add_argument("--ckpt_directory", type=str, default="/data/rohith/captain_cook/checkpoints", help="checkpoint directory")
        parser.add_argument("--split", type=str, default=const.RECORDINGS_SPLIT, help="split")
        parser.add_argument("--variant", type=str, default=const.TRANSFORMER_VARIANT, help="variant")
        parser.add_argument("--model_name", type=str, default=None, help="model name")
        parser.add_argument("--task_name", type=str, default=const.ERROR_RECOGNITION, help="task name")
        parser.add_argument("--error_category", type=str, help="error category")
        parser.add_argument("--modality", type=str, nargs="+", default=[const.VIDEO], help="audio")

        return parser

    def set_model_name(self, model_name):
        self.model_name = model_name

    def print_config(self):
        """
        Prints the configuration
        :return:
        """
        print("Configuration:")
        for k, v in self.__dict__.items():
            print(f"{k}: {v}")
        print("\\n")
''')

    # 3. dataloader/CaptainCookStepDataset.py
    with open('dataloader/CaptainCookStepDataset.py', 'w') as f:
        f.write('''import json
import math
import os

import numpy as np
import torch
from torch.utils.data import Dataset
from constants import Constants as const


class CaptainCookStepDataset(Dataset):

    def __init__(self, config, phase, split):
        self._config = config
        self._backbone = self._config.backbone
        self._phase = phase
        self._split = split

        self._modality = config.modality

        with open('annotations/annotation_json/step_annotations.json', 'r') as f:
            self._annotations = json.load(f)

        with open('annotations/annotation_json/error_annotations.json', 'r') as f:
            self._error_annotations = json.load(f)

        print("Loaded annotations...... ")

        assert self._phase in ["train", "val", "test"], f"Invalid phase: {self._phase}"

        self._build_error_category_label_name_map()
        self._build_error_category_labels()

        if self._split == const.STEP_SPLIT:
            self._init_step_split(config, phase)
        else:
            self._init_other_split_from_file(config, phase)

    def _build_error_category_label_name_map(self):
        self._error_category_name_label_map = {const.TECHNIQUE_ERROR: 6, const.PREPARATION_ERROR: 2,
                                               const.TEMPERATURE_ERROR: 3, const.MEASUREMENT_ERROR: 4,
                                               const.TIMING_ERROR: 5}

        self._error_category_label_name_map = {6: const.TECHNIQUE_ERROR, 2: const.PREPARATION_ERROR,
                                               3: const.TEMPERATURE_ERROR, 4: const.MEASUREMENT_ERROR,
                                               5: const.TIMING_ERROR}

        self._category_name_map = {
            'TechniqueError': const.TECHNIQUE_ERROR,
            'PreparationError': const.PREPARATION_ERROR,
            'TemperatureError': const.TEMPERATURE_ERROR,
            'MeasurementError': const.MEASUREMENT_ERROR,
            'TimingError': const.TIMING_ERROR
        }

    def _build_error_category_labels(self):
        self._recording_step_error_labels = {}
        for recording_step_dictionary in self._error_annotations:
            recording_id = recording_step_dictionary['recording_id']
            self._recording_step_error_labels[recording_id] = {}
            for step_annotation_dict in recording_step_dictionary['step_annotations']:
                step_id = step_annotation_dict['step_id']
                self._recording_step_error_labels[recording_id][step_id] = set()
                if "errors" not in step_annotation_dict:
                    self._recording_step_error_labels[recording_id][step_id].add(0)
                else:
                    for error_dict in step_annotation_dict['errors']:
                        error_tag = error_dict['tag']
                        if error_tag in self._error_category_name_label_map:
                            error_label = self._error_category_name_label_map[error_tag]
                        else:
                            error_label = 0

                        assert error_label is not None, f"Error label not found for error_tag: {error_tag}"
                        self._recording_step_error_labels[recording_id][step_id].add(error_label)

    def _prepare_recording_step_dictionary(self, recording_id):
        recording_step_dictionary = {}
        for step in self._annotations[recording_id]['steps']:
            step_start_time = step['start_time']
            step_end_time = step['end_time']
            step_id = step['step_id']
            if step_start_time < 0 or step_end_time < 0:
                # Ignore missing steps
                continue
            error_category_labels = self._recording_step_error_labels[recording_id][step_id]

            if recording_step_dictionary.get(step_id) is None:
                recording_step_dictionary[step_id] = []

            recording_step_dictionary[step_id].append(
                (math.floor(step_start_time), math.ceil(step_end_time), step['has_errors'], error_category_labels))
        return recording_step_dictionary

    def _init_step_split(self, config, phase):
        self._recording_ids_file = "recordings_combined_splits.json"
        print(f"Loading recording ids from {self._recording_ids_file}")
        # annotations_file_path = os.path.join(os.path.dirname(__file__), f'../er_annotations/{self._recording_ids_file}')
        annotations_file_path = f"./er_annotations/{self._recording_ids_file}"
        with open(f'{annotations_file_path}', 'r') as file:
            self._recording_ids_json = json.load(file)

        self._recording_ids = self._recording_ids_json['train'] + self._recording_ids_json['val'] + \
                              self._recording_ids_json['test']

        self._step_dict = {}
        step_index_id = 0
        for recording_id in self._recording_ids:
            self._normal_step_dict = {}
            self._error_step_dict = {}
            normal_index_id = 0
            error_index_id = 0
            # 1. Prepare step_id, list(<start, end>) for the recording_id
            recording_step_dictionary = self._prepare_recording_step_dictionary(recording_id)

            # 2. Add step start and end time list to the step_dict
            for step_id in recording_step_dictionary.keys():
                # If the step has errors, add it to the error_step_dict, else add it to the normal_step_dict
                if recording_step_dictionary[step_id][0][2]:
                    self._error_step_dict[f'E{error_index_id}'] = (recording_id, recording_step_dictionary[step_id])
                    error_index_id += 1
                else:
                    self._normal_step_dict[f'N{normal_index_id}'] = (
                        recording_id, recording_step_dictionary[step_id])
                    normal_index_id += 1

            np.random.seed(config.seed)
            np.random.shuffle(list(self._normal_step_dict.keys()))
            np.random.shuffle(list(self._error_step_dict.keys()))

            normal_step_indices = list(self._normal_step_dict.keys())
            error_step_indices = list(self._error_step_dict.keys())

            self._split_proportion = [0.75, 0.16, 0.9]

            num_normal_steps = len(normal_step_indices)
            num_error_steps = len(error_step_indices)

            self._split_proportion_normal = [int(num_normal_steps * self._split_proportion[0]),
                                             int(num_normal_steps * (
                                                     self._split_proportion[0] + self._split_proportion[1]))]
            self._split_proportion_error = [int(num_error_steps * self._split_proportion[0]),
                                            int(num_error_steps * (
                                                    self._split_proportion[0] + self._split_proportion[1]))]

            if phase == 'train':
                self._train_normal = normal_step_indices[:self._split_proportion_normal[0]]
                self._train_error = error_step_indices[:self._split_proportion_error[0]]
                train_indices = self._train_normal + self._train_error
                for index_id in train_indices:
                    self._step_dict[step_index_id] = self._normal_step_dict.get(index_id,
                                                                                self._error_step_dict.get(index_id))
                    step_index_id += 1
            elif phase == 'test':
                self._val_normal = normal_step_indices[
                                   self._split_proportion_normal[0]:self._split_proportion_normal[1]]
                self._val_error = error_step_indices[
                                  self._split_proportion_error[0]:self._split_proportion_error[1]]
                val_indices = self._val_normal + self._val_error
                for index_id in val_indices:
                    self._step_dict[step_index_id] = self._normal_step_dict.get(index_id,
                                                                                self._error_step_dict.get(index_id))
                    step_index_id += 1
            elif phase == 'val':
                self._test_normal = normal_step_indices[self._split_proportion_normal[1]:]
                self._test_error = error_step_indices[self._split_proportion_error[1]:]
                test_indices = self._test_normal + self._test_error
                for index_id in test_indices:
                    self._step_dict[step_index_id] = self._normal_step_dict.get(index_id,
                                                                                self._error_step_dict.get(index_id))
                    step_index_id += 1

    def _init_other_split_from_file(self, config, phase):
        self._recording_ids_file = f"{self._split}_combined_splits.json"
        annotations_file_path = f"./er_annotations/{self._recording_ids_file}"
        print(f"Loading recording ids from {self._recording_ids_file}")
        with open(f'{annotations_file_path}', 'r') as file:
            self._recording_ids_json = json.load(file)

        self._recording_ids = self._recording_ids_json[phase]
        self._step_dict = {}
        index_id = 0
        for recording_id in self._recording_ids:
            # 1. Prepare step_id, list(<start, end>) for the recording_id
            recording_step_dictionary = self._prepare_recording_step_dictionary(recording_id)

            # 2. Add step start and end time list to the step_dict
            for step_id in recording_step_dictionary.keys():
                self._step_dict[index_id] = (recording_id, recording_step_dictionary[step_id])
                index_id += 1

    def __len__(self):
        assert len(self._step_dict) > 0, "No data found in the dataset"
        return len(self._step_dict)

    def _build_task_specific_features_labels(self, step_features, step_has_errors, step_error_category_labels):
        N, d = step_features.shape
        if self._config.task_name == const.ERROR_RECOGNITION:
            if step_has_errors:
                step_labels = torch.ones(N, 1)
            else:
                step_labels = torch.zeros(N, 1)
            return step_features, step_labels
        elif self._config.task_name == const.EARLY_ERROR_RECOGNITION:
            # Input only half of the step features and labels
            step_features = step_features[:N // 2, :]
            if step_has_errors:
                step_labels = torch.ones(N // 2, 1)
            else:
                step_labels = torch.zeros(N // 2, 1)
            return step_features, step_labels
        elif self._config.task_name == const.ERROR_CATEGORY_RECOGNITION:
            # print(f"Error category: {self._config.error_category}")
            error_category_name = self._category_name_map[self._config.error_category]
            # print(f"Error category name: {error_category_name}")
            task_error_category_label = self._error_category_name_label_map[error_category_name]
            if task_error_category_label in step_error_category_labels:
                step_labels = torch.ones(N, 1)
            else:
                step_labels = torch.zeros(N, 1)
            return step_features, step_labels

    def _build_modality_step_features_labels(self, recording_features, step_start_end_list):
        # Build step features by concatenating the features of the step from the list
        step_features = []
        step_has_errors = None
        step_error_category_labels = None
        for step_start_time, step_end_time, has_errors, error_category_labels in step_start_end_list:
            sub_step_features = recording_features[step_start_time:step_end_time, :]
            step_features.append(sub_step_features)
            step_has_errors = has_errors
            step_error_category_labels = error_category_labels
        step_features = np.concatenate(step_features, axis=0)
        step_features = torch.from_numpy(step_features).float()

        step_features, step_labels = self._build_task_specific_features_labels(
            step_features,
            step_has_errors,
            step_error_category_labels
        )

        return step_features, step_labels

    def _get_video_features(self, recording_id, step_start_end_list):
        if self._backbone == const.EGOVLP:
            base_dir = os.path.join(self._config.segment_features_directory, "features", "egovlp")
            features_path = os.path.join(base_dir, f'{recording_id}.npz')

            # If exact match not found, look for files starting with recording_id
            if not os.path.exists(features_path):
                try:
                    candidates = [f for f in os.listdir(base_dir) if f.startswith(f"{recording_id}_") and f.endswith(".npz")]
                    if candidates:
                        features_path = os.path.join(base_dir, candidates[0])
                except OSError:
                    pass

            features_data = np.load(features_path)
            recording_features = features_data['video_features']

            # --- ROBUSTNESS FIX: Handle NaNs and Normalize ---
            recording_features = np.nan_to_num(recording_features, nan=0.0, posinf=0.0, neginf=0.0)

            # Normalize (StandardScaler)
            mean = np.mean(recording_features, axis=0, keepdims=True)
            std = np.std(recording_features, axis=0, keepdims=True)
            std[std < 1e-6] = 1.0
            recording_features = (recording_features - mean) / std

            # Clip outliers
            recording_features = np.clip(recording_features, -10.0, 10.0)
            # -------------------------------------------------

        else:
            features_path = os.path.join(self._config.segment_features_directory, "video", self._backbone,
                                             f'{recording_id}_360p.mp4_1s_1s.npz')
            features_data = np.load(features_path)
            recording_features = features_data['arr_0']

        step_features, step_labels = self._build_modality_step_features_labels(recording_features, step_start_end_list)
        features_data.close()
        return step_features, step_labels

    def __getitem__(self, idx):
        recording_id = self._step_dict[idx][0]
        step_start_end_list = self._step_dict[idx][1]

        step_features = None
        step_labels = None

        assert self._backbone in [const.OMNIVORE, const.SLOWFAST, const.EGOVLP], "Only Omnivore, SlowFast and EgoVLP are supported with this codebase"
        step_features, step_labels = self._get_video_features(recording_id, step_start_end_list)

        assert step_features is not None, f"Features not found for recording_id: {recording_id}"
        assert step_labels is not None, f"Labels not found for recording_id: {recording_id}"

        return step_features, step_labels


def collate_fn(batch):
    # batch is a list of tuples, and each tuple is (step_features, step_labels)
    step_features, step_labels = zip(*batch)

    # Stack the step_features and step_labels
    step_features = torch.cat(step_features, dim=0)
    step_labels = torch.cat(step_labels, dim=0)

    return step_features, step_labels
''')

    # 4. core/models/blocks.py (Modified for EgoVLP)
    with open('core/models/blocks.py', 'w') as f:
        f.write('''import math

import torch
import torch.nn as nn
from torch import Tensor
from constants import Constants as const

# define the transformer backbone here
EncoderLayer = nn.TransformerEncoderLayer
Encoder = nn.TransformerEncoder


def fetch_input_dim(config, decoder=False):
    if config.backbone == const.OMNIVORE:
        return 1024
    elif config.backbone == const.SLOWFAST:
        return 400
    elif config.backbone == const.X3D:
        return 400
    elif config.backbone == const.RESNET3D:
        return 400
    elif config.backbone == const.EGOVLP:
        return 256 # Assuming 256 dim for EgoVLP
    elif config.backbone == const.IMAGEBIND:
        if decoder is True:
            return 1024
        k = len(config.modality)
        return 1024 * k



class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLP, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = self.layer2(x)
        return x


class MLP1(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLP1, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size * 8)
        self.layer2 = nn.Linear(hidden_size * 8, hidden_size * 2)
        self.layer3 = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = self.layer3(x)
        return x


class CNN(nn.Module):
    def __init__(self, in_channels, final_width, final_height, num_classes):
        super(CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * final_width * final_height, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: Tensor, indices=None) -> Tensor:
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        if indices is None:
            x = x + self.pe[:, :x.size(1)]
        else:
            pos = torch.cat([self.pe[:, index] for index in indices])
            x = x + pos
        return self.dropout(x)


class PositionalEncodingLearn(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        self.embed = nn.Embedding(max_len, d_model)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.uniform_(self.embed.weight)

    def forward(self, x, indices=None):
        # x: b, l, d
        r = torch.arange(x.shape[1], device=x.device)
        embed = self.embed(r)
        return x + embed.repeat(x.shape[0], 1, 1)
''')

    # 5. core/models/er_former.py
    with open('core/models/er_former.py', 'w') as f:
        f.write('''import torch
from torch import nn

from core.models.blocks import EncoderLayer, Encoder, MLP, fetch_input_dim


class ErFormer(nn.Module):

    def __init__(self, config, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.config = config
        input_dimension = fetch_input_dim(config)

        # Initialize the transformer encoder
        step_encoder_layer = EncoderLayer(d_model=input_dimension, dim_feedforward=2048, nhead=8, batch_first=True)
        self.step_encoder = Encoder(step_encoder_layer, num_layers=1)
        decoder_input_dimension = fetch_input_dim(config, decoder=True)
        # Initialize the MLP decoder
        self.decoder = MLP(decoder_input_dimension, 512, 1)
        self.apply(init_weights)  # Apply weight initialization

    def forward(self, input_data):
        # Check for NaNs in input and replace them with zero
        input_data = torch.nan_to_num(input_data, nan=0.0, posinf=1.0, neginf=-1.0)

        # Encode the input
        encoded_output = self.step_encoder(input_data)
        _, dim = encoded_output.shape

        audio_output = None
        text_output = None
        depth_output = None

        # Split the encoded output into video, audio, text and depth outputs
        # Modality Order: Video, Audio, Text, Depth
        video_output = encoded_output[:, :1024]
        if dim // 1024 == 1:
            video_output = encoded_output[:, :1024]
        elif dim // 1024 == 2:
            video_output = encoded_output[:, :1024]
            audio_output = encoded_output[:, 1024:2048]
        elif dim // 1024 == 3:
            video_output = encoded_output[:, :1024]
            audio_output = encoded_output[:, 1024:2048]
            text_output = encoded_output[:, 2048:3072]
        elif dim // 1024 == 4:
            video_output = encoded_output[:, :1024]
            audio_output = encoded_output[:, 1024:2048]
            text_output = encoded_output[:, 2048:3072]
            depth_output = encoded_output[:, 3072:]

        # Do a weighted sum of the outputs
        if dim // 1024 == 1:
            encoded_output = video_output
        elif dim // 1024 == 2:
            encoded_output = 0.65 * video_output + 0.35 * audio_output
        elif dim // 1024 == 3:
            encoded_output = 0.4 * video_output + 0.3 * audio_output + 0.3 * text_output
        elif dim // 1024 == 4:
            encoded_output = 0.25 * video_output + 0.25 * audio_output + 0.25 * text_output + 0.25 * depth_output

        # Decode the output
        final_output = self.decoder(encoded_output)

        # Check for NaNs in output and replace them with zero
        # final_output = torch.nan_to_num(final_output, nan=0.0, posinf=1.0, neginf=-1.0)

        return final_output

def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.LayerNorm):
        torch.nn.init.constant_(m.bias, 0)
        torch.nn.init.constant_(m.weight, 1.0)
''')

    # 6. base.py
    with open('base.py', 'w') as f:
        f.write('''import csv
import os

from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau

import wandb
from torch import optim, nn
from torch.utils.data import DataLoader

from constants import Constants as const
import numpy as np
import torch
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
from torcheval.metrics.functional import binary_auprc
from tqdm import tqdm

from core.models.blocks import fetch_input_dim, MLP
from core.models.er_former import ErFormer
from dataloader.CaptainCookStepDataset import collate_fn, CaptainCookStepDataset
from dataloader.CaptainCookSubStepDataset import CaptainCookSubStepDataset


def fetch_model_name(config):
    if config.task_name == const.ERROR_CATEGORY_RECOGNITION:
        return fetch_model_name_ecr(config)
    elif config.task_name in  [const.EARLY_ERROR_RECOGNITION, const.ERROR_RECOGNITION]:
        if config.model_name is None:
            if config.backbone in [const.RESNET3D, const.X3D, const.SLOWFAST, const.OMNIVORE]:
                config.model_name = f"{config.task_name}_{config.split}_{config.backbone}_{config.variant}_{config.modality[0]}"
            elif config.backbone == const.IMAGEBIND:
                combined_modality_name = '_'.join(config.modality)
                config.model_name = f"{config.task_name}_{config.split}_{config.backbone}_{config.variant}_{combined_modality_name}"
    return config.model_name


def fetch_model_name_ecr(config):
    combined_modality_name = '_'.join(config.modality)
    if config.model_name is None:
        config.model_name = (f"{config.task_name}_{config.split}_{config.backbone}"
                             f"_{config.variant}_{combined_modality_name}_{config.error_category}")
    return config.model_name


def fetch_model(config):
    model = None
    if config.variant == const.MLP_VARIANT:
        if config.backbone in [const.OMNIVORE, const.RESNET3D, const.X3D, const.SLOWFAST, const.IMAGEBIND]:
            input_dim = fetch_input_dim(config)
            model = MLP(input_dim, 512, 1)
    elif config.variant == const.TRANSFORMER_VARIANT:
        if config.backbone in [const.OMNIVORE, const.RESNET3D, const.X3D, const.SLOWFAST, const.IMAGEBIND, const.EGOVLP]:
            model = ErFormer(config)

    assert model is not None, f"Model not found for variant: {config.variant} and backbone: {config.backbone}"
    model.to(config.device)
    return model


def convert_and_round(value):
    value = value * 100.0
    if isinstance(value, torch.Tensor):
        return np.round(value.numpy(), 2)
    return np.round(value, 2)


def collate_stats(config, sub_step_metrics, step_metrics):
    collated_stats = [config.split, config.backbone, config.variant, config.modality]
    for metric in [const.PRECISION, const.RECALL, const.F1, const.ACCURACY, const.AUC, const.PR_AUC]:
        collated_stats.append(convert_and_round(sub_step_metrics[metric]))
    for metric in [const.PRECISION, const.RECALL, const.F1, const.ACCURACY, const.AUC, const.PR_AUC]:
        # Round to two digits before appending
        collated_stats.append(convert_and_round(step_metrics[metric]))
    return collated_stats


def save_results_to_csv(config, sub_step_metrics, step_metrics, step_normalization=False, sub_step_normalization=False,
                        threshold=0.5):
    results_dir = os.path.join(os.getcwd(), const.RESULTS)
    task_results_dir = os.path.join(results_dir, config.task_name, "combined_results")
    os.makedirs(task_results_dir, exist_ok=True)
    config.model_name = fetch_model_name(config)

    results_file_path = os.path.join(task_results_dir,
                                     f'step_{step_normalization}_substep_{sub_step_normalization}_threshold_{threshold}.csv')
    collated_stats = collate_stats(config, sub_step_metrics, step_metrics)

    file_exist = os.path.isfile(results_file_path)

    with open(results_file_path, "a", newline='') as activity_idx_step_idx_annotation_csv_file:
        writer = csv.writer(activity_idx_step_idx_annotation_csv_file, quoting=csv.QUOTE_NONNUMERIC)
        if not file_exist:
            writer.writerow([
                "Split", "Backbone", "Variant", "Modality",
                "Sub-Step Precision", "Sub-Step Recall", "Sub-Step F1", "Sub-Step Accuracy", "Sub-Step AUC",
                "Sub-Step PR AUC",
                "Step Precision", "Step Recall", "Step F1", "Step Accuracy", "Step AUC", "Step PR AUC"
            ])
        writer.writerow(collated_stats)


def train_model_base(train_loader, val_loader, config, test_loader=None):
    model = fetch_model(config)
    optimizer = optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = ReduceLROnPlateau(optimizer, 'min', patience=2)

    best_val_loss = float('inf')
    best_model_state = None

    for epoch in range(config.num_epochs):
        model.train()
        train_loss = 0
        valid_batches = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{config.num_epochs}")
        for data, target in pbar:
            data, target = data.to(config.device), target.to(config.device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)

            if torch.isnan(loss):
                print(f"Warning: NaN loss detected at epoch {epoch + 1}. Input stats: Min={data.min():.2f}, Max={data.max():.2f}")
                optimizer.zero_grad()
                continue

            loss.backward()

            # Gradient Clipping & Check
            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if torch.isnan(total_norm):
                 print("Warning: NaN gradient norm detected. Skipping step to avoid corrupting weights.")
                 optimizer.zero_grad()
                 continue

            optimizer.step()

            train_loss += loss.item()
            valid_batches += 1
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        if valid_batches > 0:
            train_loss /= valid_batches
        else:
            train_loss = float('nan')
        print(f"Train Loss: {train_loss:.4f}")

        # Validation
        model.eval()
        val_loss = 0
        valid_val_batches = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(config.device), target.to(config.device)
                output = model(data)
                loss = criterion(output, target)
                if not torch.isnan(loss):
                    val_loss += loss.item()
                    valid_val_batches += 1

        if valid_val_batches > 0:
            val_loss /= valid_val_batches
        else:
            val_loss = float('nan')
        print(f"Val Loss: {val_loss:.4f}")

        if not torch.isnan(torch.tensor(val_loss)):
            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = model.state_dict()
                if config.save_model:
                    os.makedirs(config.ckpt_directory, exist_ok=True)
                    torch.save(model.state_dict(), os.path.join(config.ckpt_directory, f"{config.model_name}.pth"))

    # Test with best model
    if test_loader:
        print("Testing with best model...")
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            test_er_model(model, test_loader, criterion, config.device, "test")
        else:
            print("WARNING: No valid model checkpoint found. Skipping testing.")


def train_step_test_step_dataset_base(config):
    cuda_kwargs = {'num_workers': 0, 'pin_memory': True} if config.device == "cuda" else {}
    train_kwargs = {**cuda_kwargs, "shuffle": True, "batch_size": config.batch_size}
    test_kwargs = {**cuda_kwargs, "shuffle": False, "batch_size": config.test_batch_size}

    train_dataset = CaptainCookStepDataset(config, const.TRAIN, config.split)
    train_loader = DataLoader(train_dataset, collate_fn=collate_fn, **train_kwargs)

    val_dataset = CaptainCookStepDataset(config, const.VAL, config.split)
    val_loader = DataLoader(val_dataset, collate_fn=collate_fn, **test_kwargs)

    test_dataset = CaptainCookStepDataset(config, const.TEST, config.split)
    test_loader = DataLoader(test_dataset, collate_fn=collate_fn, **test_kwargs)

    return train_loader, val_loader, test_loader


def train_sub_step_test_step_dataset_base(config):
    cuda_kwargs = {'num_workers': 0, 'pin_memory': True} if config.device == "cuda" else {}
    train_kwargs = {**cuda_kwargs, "shuffle": True, "batch_size": 1024}
    test_kwargs = {**cuda_kwargs, "shuffle": False, "batch_size": 1}

    train_dataset = CaptainCookSubStepDataset(config, const.TRAIN, config.split)
    train_loader = DataLoader(train_dataset, collate_fn=collate_fn, **train_kwargs)
    val_dataset = CaptainCookStepDataset(config, const.TEST, config.split)
    val_loader = DataLoader(val_dataset, collate_fn=collate_fn, **test_kwargs)
    test_dataset = CaptainCookStepDataset(config, const.TEST, config.split)
    test_loader = DataLoader(test_dataset, collate_fn=collate_fn, **test_kwargs)

    print("-------------------------------------------------------------")
    print("Training sub-step model and testing on step level")
    print(f"Train args: {train_kwargs}")
    print(f"Test args: {test_kwargs}")
    print(f"Split: {config.split}")
    print("-------------------------------------------------------------")

    return train_loader, val_loader, test_loader


# ----------------------- TEST BASE FILES -----------------------


def test_er_model(model, test_loader, criterion, device, phase, step_normalization=True, sub_step_normalization=True,
                  threshold=0.6):
    total_samples = 0
    all_targets = []
    all_outputs = []

    test_loader = tqdm(test_loader)
    num_batches = len(test_loader)
    test_losses = []

    test_step_start_end_list = []
    counter = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            total_samples += data.shape[0]
            loss = criterion(output, target)

            if not torch.isnan(loss):
                test_losses.append(loss.item())

            sigmoid_output = output.sigmoid()
            all_outputs.append(sigmoid_output.detach().cpu().numpy().reshape(-1))
            all_targets.append(target.detach().cpu().numpy().reshape(-1))

            test_step_start_end_list.append((counter, counter + data.shape[0]))
            counter += data.shape[0]

            # Set the description of the tqdm instance to show the loss
            test_loader.set_description(f'{phase} Progress: {total_samples}/{num_batches}')

    # Flatten lists
    all_outputs = np.concatenate(all_outputs)
    all_targets = np.concatenate(all_targets)

    # Assert that none of the outputs are NaN
    if np.isnan(all_outputs).any():
        print("WARNING: NaNs found in model outputs. Replacing with 0.")
        all_outputs = np.nan_to_num(all_outputs, nan=0.0)

    # ------------------------- Sub-Step Level Metrics -------------------------
    all_sub_step_targets = all_targets.copy()
    all_sub_step_outputs = all_outputs.copy()

    # Calculate metrics at the sub-step level
    pred_sub_step_labels = (all_sub_step_outputs > 0.5).astype(int)
    sub_step_precision = precision_score(all_sub_step_targets, pred_sub_step_labels, zero_division=0)
    sub_step_recall = recall_score(all_sub_step_targets, pred_sub_step_labels, zero_division=0)
    sub_step_f1 = f1_score(all_sub_step_targets, pred_sub_step_labels, zero_division=0)
    sub_step_accuracy = accuracy_score(all_sub_step_targets, pred_sub_step_labels)
    try:
        sub_step_auc = roc_auc_score(all_sub_step_targets, all_sub_step_outputs)
    except ValueError:
        sub_step_auc = 0.0
    sub_step_pr_auc = binary_auprc(torch.tensor(pred_sub_step_labels), torch.tensor(all_sub_step_targets))

    sub_step_metrics = {
        const.PRECISION: sub_step_precision,
        const.RECALL: sub_step_recall,
        const.F1: sub_step_f1,
        const.ACCURACY: sub_step_accuracy,
        const.AUC: sub_step_auc,
        const.PR_AUC: sub_step_pr_auc
    }

    # -------------------------- Step Level Metrics --------------------------
    all_step_targets = []
    all_step_outputs = []

    # threshold_outputs = all_outputs / max_probability

    for start, end in test_step_start_end_list:
        step_output = all_outputs[start:end]
        step_target = all_targets[start:end]

        step_output = np.array(step_output)

        if len(step_output) == 0:
            mean_step_output = 0.0
            target_val = 0
        else:
            # # Scale the output to [0, 1]
            if start - end > 1:
                if sub_step_normalization:
                    prob_range = np.max(step_output) - np.min(step_output)
                    if prob_range > 0:
                        step_output = (step_output - np.min(step_output)) / prob_range

            mean_step_output = np.mean(step_output)
            target_val = 1 if np.mean(step_target) > 0.95 else 0

        all_step_outputs.append(mean_step_output)
        all_step_targets.append(target_val)

    all_step_outputs = np.array(all_step_outputs)

    if np.isnan(all_step_outputs).any():
        all_step_outputs = np.nan_to_num(all_step_outputs, nan=0.0)

    # # Scale the output to [0, 1]
    if step_normalization:
        prob_range = np.max(all_step_outputs) - np.min(all_step_outputs)
        if prob_range > 0:
            all_step_outputs = (all_step_outputs - np.min(all_step_outputs)) / prob_range

    all_step_targets = np.array(all_step_targets)

    # Calculate metrics at the step level
    pred_step_labels = (all_step_outputs > threshold).astype(int)
    precision = precision_score(all_step_targets, pred_step_labels, zero_division=0)
    recall = recall_score(all_step_targets, pred_step_labels, zero_division=0)
    f1 = f1_score(all_step_targets, pred_step_labels, zero_division=0)
    accuracy = accuracy_score(all_step_targets, pred_step_labels)

    try:
        auc = roc_auc_score(all_step_targets, all_step_outputs)
    except ValueError:
        auc = 0.0
    pr_auc = binary_auprc(torch.tensor(pred_step_labels), torch.tensor(all_step_targets))

    step_metrics = {
        const.PRECISION: precision,
        const.RECALL: recall,
        const.F1: f1,
        const.ACCURACY: accuracy,
        const.AUC: auc,
        const.PR_AUC: pr_auc
    }

    # Print step level metrics
    print("----------------------------------------------------------------")
    print(f'{phase} Sub Step Level Metrics: {sub_step_metrics}')
    print(f"{phase} Step Level Metrics: {step_metrics}")
    print("----------------------------------------------------------------")

    return test_losses, sub_step_metrics, step_metrics
''')

    # 7. train_er.py
    with open('train_er.py', 'w') as f:
        f.write('''import wandb
from base import fetch_model_name, train_step_test_step_dataset_base, train_sub_step_test_step_dataset_base, \
    train_model_base
from core.config import Config
from core.utils import init_logger_and_wandb
from constants import Constants as const


def train_sub_step_test_step_er(config):
    train_loader, val_loader, test_loader = train_sub_step_test_step_dataset_base(config)
    train_model_base(train_loader, val_loader, config)


def train_step_test_step_er(config):
    train_loader, val_loader, test_loader = train_step_test_step_dataset_base(config)
    train_model_base(train_loader, val_loader, config, test_loader=test_loader)


def main():
    conf = Config()
    conf.task_name = const.ERROR_RECOGNITION
    if conf.model_name is None:
        m_name = fetch_model_name(conf)
        conf.model_name = m_name

    if conf.enable_wandb:
        init_logger_and_wandb(conf)

    train_step_test_step_er(conf)

    if conf.enable_wandb:
        wandb.finish()


if __name__ == "__main__":
    main()
''')

    # Install requirements
    print("Installing requirements...")
    # NOTE: Skipping requirements.txt as per instruction to use default Colab torch
    # !pip install -r requirements.txt

    # We MUST install torcheval as it is missing in Colab but required by core.evaluate
    !pip install torcheval loguru

except ImportError:
    IN_COLAB = False
    print("Not running in Colab")

# --- IMPORTS ---
print("Importing modules...")
import torch
import numpy as np
from core.config import Config
# from core.evaluate import evaluate # REMOVED: Not needed and causes import error
from core.models.er_former import ErFormer # Changed to ErFormer (capital E, lowercase r)
from dataloader.CaptainCookStepDataset import CaptainCookStepDataset, collate_fn
from constants import Constants as const
from train_er import train_step_test_step_er # ADDED: Import the training function

print("Setup complete!")

Running in Google Colab
Mounted at /content/drive
Cloning repository from https://github.com/zeynepoztunc/aml-procedural-mistake-detection.git...
Cloning into '/content/code'...
remote: Enumerating objects: 443, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 443 (delta 8), reused 7 (delta 7), pack-reused 430 (from 2)
Receiving objects: 100% (443/443), 118.23 KiB | 1000.00 KiB/s, done.
Resolving deltas: 100% (289/289), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 3.09 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodul

In [2]:
# 4. Setup Data from Drive (Colab Only)
if IN_COLAB:
    drive.mount('/content/drive')

    # Configuration for Drive paths
    # Based on your previous notebook: DRIVE_BASE_PATH = "/content/drive/MyDrive/AML_Project"
    DRIVE_BASE_PATH = "/content/drive/MyDrive/AML_Project"
    DRIVE_FEATURE_DIR = f"{DRIVE_BASE_PATH}/features/egovlp"

    # Local destination
    LOCAL_FEATURE_DIR = "data/features/egovlp"

    if not os.path.exists(LOCAL_FEATURE_DIR):
        print(f"Creating directory {LOCAL_FEATURE_DIR}...")
        os.makedirs(LOCAL_FEATURE_DIR, exist_ok=True)

        if os.path.exists(DRIVE_FEATURE_DIR):
            print(f"Copying features from {DRIVE_FEATURE_DIR}...")
            !cp -r "{DRIVE_FEATURE_DIR}"/*.npz "{LOCAL_FEATURE_DIR}/"
            print(f"Copied features to {LOCAL_FEATURE_DIR}")
        else:
            print(f"WARNING: Drive directory {DRIVE_FEATURE_DIR} not found!")
    else:
        print(f"Features directory {LOCAL_FEATURE_DIR} already exists.")

    # Verify
    if os.path.exists(LOCAL_FEATURE_DIR):
        print(f"Found {len(os.listdir(LOCAL_FEATURE_DIR))} files in local feature directory.")
else:
    # Local check
    LOCAL_FEATURE_DIR = "data/features/egovlp"
    if os.path.exists(LOCAL_FEATURE_DIR):
        print(f"Found {len(os.listdir(LOCAL_FEATURE_DIR))} files in {LOCAL_FEATURE_DIR}")
    else:
        print(f"WARNING: {LOCAL_FEATURE_DIR} not found. Please download features.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Creating directory data/features/egovlp...
Copying features from /content/drive/MyDrive/AML_Project/features/egovlp...
Copied features to data/features/egovlp
Found 384 files in local feature directory.


In [3]:
import sys

# Configuration
# We override the default configuration to use EgoVLP
class NotebookConfig(Config):
    def __init__(self):
        # Temporarily save and modify sys.argv to prevent argparse from failing in Colab
        original_argv = sys.argv
        sys.argv = [original_argv[0]] # Keep the script name, but remove all other args

        super().__init__()

        # Restore sys.argv
        sys.argv = original_argv

        # Override defaults
        self.backbone = const.EGOVLP
        self.modality = [const.VIDEO]
        self.task_name = const.ERROR_RECOGNITION
        self.variant = const.TRANSFORMER_VARIANT # V2 Baseline
        self.num_epochs = 20 # Adjust as needed
        self.batch_size = 8
        self.lr = 1e-4
        self.enable_wandb = False # Set to True if you have wandb setup
        self.segment_features_directory = "data/" # Root for features

        # Ensure device is correct
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")

config = NotebookConfig()
config.print_config()


Using device: cuda
Configuration:
backbone: egovlp
modality: ['video']
phase: train
segment_length: 1
segment_features_directory: data/
ckpt_directory: /data/rohith/captain_cook/checkpoints
split: recordings
batch_size: 8
test_batch_size: 1
num_epochs: 20
lr: 0.0001
weight_decay: 0.001
log_interval: 5
dry_run: False
ckpt: None
seed: 42
device: cuda
variant: Transformer
model_name: None
task_name: error_recognition
error_category: None
enable_wandb: False
parser: ArgumentParser(prog='colab_kernel_launcher.py', usage=None, description='training code', formatter_class=<class 'argparse.HelpFormatter'>, conflict_handler='error', add_help=True)
args: {'batch_size': 1, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'egovlp', 'ckpt_directory': '/data/rohith/captain_cook/checkpoints', 'split': 'recordings', 'variant': 'Transformer', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video

In [4]:
# Run Training
# This will train the model and evaluate on the test set

from dataloader.CaptainCookStepDataset import CaptainCookStepDataset, collate_fn
from constants import Constants as const
from base import fetch_model, test_er_model
from tqdm import tqdm
from torch import optim, nn
import os
import numpy as np
import torch

# --- CONFIGURATION ---
config.batch_size = 8
config.test_batch_size = 1
# Reduce LR further for stability
config.lr = 5e-6
print(f"Training Config -> LR: {config.lr}, Batch Size: {config.batch_size}")

# --- 1. ROBUST DATA LOADER PATCH ---

def _corrected_get_video_features(self, recording_id, step_start_end_list):
    # Construct the correct path format: {id}_360p_224.npz
    if self._backbone == const.EGOVLP:
        filename = f'{recording_id}_360p_224.npz'
        features_path = os.path.join(self._config.segment_features_directory, "features", "egovlp", filename)
    else:
        features_path = os.path.join(self._config.segment_features_directory, "video", self._backbone,
                                     f'{recording_id}_360p.mp4_1s_1s.npz')

    if os.path.exists(features_path):
        try:
            features_data = np.load(features_path)
            if 'video_features' in features_data:
                recording_features = features_data['video_features']
            else:
                recording_features = features_data['arr_0']
            features_data.close()

            # --- SANITIZATION & NORMALIZATION ---
            recording_features = recording_features.astype(np.float32)
            recording_features = np.nan_to_num(recording_features, nan=0.0, posinf=0.0, neginf=0.0)

            # Normalize (StandardScaler)
            mean = np.mean(recording_features, axis=0, keepdims=True)
            std = np.std(recording_features, axis=0, keepdims=True)
            std[std < 1e-6] = 1.0
            recording_features = (recording_features - mean) / std

            # Clip outliers
            recording_features = np.clip(recording_features, -10.0, 10.0)

            step_features, step_labels = self._build_modality_step_features_labels(recording_features, step_start_end_list)
            return step_features, step_labels

        except Exception as e:
            print(f"Error reading {features_path}: {e}")

    # Fallback: Return Zeros
    dim = 256 if self._backbone == const.EGOVLP else 400
    if self._backbone == const.OMNIVORE: dim = 1024
    max_end = max([end for _, end, _, _ in step_start_end_list]) if step_start_end_list else 0
    dummy_features = np.zeros((max_end + 1, dim), dtype=np.float32)
    return self._build_modality_step_features_labels(dummy_features, step_start_end_list)

CaptainCookStepDataset._get_video_features = _corrected_get_video_features

# --- 2. WEIGHT INITIALIZATION ---
def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.LayerNorm):
        torch.nn.init.constant_(m.bias, 0)
        torch.nn.init.constant_(m.weight, 1.0)

# --- 3. ROBUST TRAINING LOOP ---

def train_model_debug(train_loader, val_loader, config, test_loader=None):
    model = fetch_model(config)

    # Apply manual weight initialization to stabilize Transformer
    model.apply(init_weights)
    print("Applied Xavier initialization to model weights.")

    model.train()

    optimizer = optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    best_val_loss = float('inf')

    for epoch in range(config.num_epochs):
        model.train()
        train_loss = 0
        batch_count = 0

        # Training
        for data, target in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{config.num_epochs}"):
            data, target = data.to(config.device), target.to(config.device)

            # Debug first batch only
            if epoch == 0 and batch_count == 0:
                print(f"\n[DEBUG] Batch 0 Stats: Min={data.min():.2f}, Max={data.max():.2f}, Mean={data.mean():.2f}")

            optimizer.zero_grad()
            output = model(data)

            # Safety check on output
            if torch.isnan(output).any():
                print(f"[WARNING] NaN in model output at batch {batch_count}. Skipping.")
                continue

            loss = criterion(output, target)

            if torch.isnan(loss):
                print(f"[WARNING] NaN Loss at batch {batch_count}. Skipping step.")
                continue

            loss.backward()

            # --- TIGHTER GRADIENT CLIPPING ---
            # Reduced from 1.0 to 0.5 for extra stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)

            optimizer.step()
            train_loss += loss.item()
            batch_count += 1

        if batch_count > 0:
            train_loss /= batch_count
            print(f"Train Loss: {train_loss:.4f}")

        # Validation
        model.eval()
        val_loss = 0
        val_batches = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(config.device), target.to(config.device)
                output = model(data)
                loss = criterion(output, target)
                if not torch.isnan(loss):
                    val_loss += loss.item()
                    val_batches += 1

        if val_batches > 0:
            val_loss /= val_batches
            print(f"Val Loss: {val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
        else:
            print("Val Loss: NaN (All batches)")

    # Test
    if test_loader:
        print("Testing...")
        test_er_model(model, test_loader, criterion, config.device, "test")

# --- 4. RUN TRAINING ---
try:
    cuda_kwargs = {'num_workers': 0, 'pin_memory': True} if config.device == "cuda" else {}
    train_kwargs = {**cuda_kwargs, "shuffle": True, "batch_size": config.batch_size}
    test_kwargs = {**cuda_kwargs, "shuffle": False, "batch_size": config.test_batch_size}

    train_dataset = CaptainCookStepDataset(config, const.TRAIN, config.split)
    train_loader = torch.utils.data.DataLoader(train_dataset, collate_fn=collate_fn, **train_kwargs)

    val_dataset = CaptainCookStepDataset(config, const.VAL, config.split)
    val_loader = torch.utils.data.DataLoader(val_dataset, collate_fn=collate_fn, **test_kwargs)

    test_dataset = CaptainCookStepDataset(config, const.TEST, config.split)
    test_loader = torch.utils.data.DataLoader(test_dataset, collate_fn=collate_fn, **test_kwargs)

    train_model_debug(train_loader, val_loader, config, test_loader=test_loader)

except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()

Training Config -> LR: 5e-06, Batch Size: 8
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Applied Xavier initialization to model weights.


Epoch 1/20:   0%|          | 0/497 [00:00<?, ?it/s]


[DEBUG] Batch 0 Stats: Min=-3.74, Max=3.47, Mean=-0.02


Epoch 1/20:  24%|██▍       | 121/497 [00:06<00:16, 23.12it/s]

[WARNING] NaN Loss at batch 116. Skipping step.


...

[WARNING] NaN Loss at batch 330. Skipping step.


Epoch 20/20: 100%|██████████| 497/497 [00:10<00:00, 47.88it/s]


Train Loss: 0.5942
Val Loss: 0.6931
Testing...


test Progress: 18818/671: 100%|██████████| 671/671 [00:03<00:00, 201.11it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.5333333333333333, 'recall': 0.001164652787887611, 'f1': 0.002324230098779779, 'accuracy': 0.6350302901477309, 'auc': np.float64(0.5828988577296672), 'pr_auc': tensor(0.3652)}
test Step Level Metrics: {'precision': 0.45689655172413796, 'recall': 0.7851851851851852, 'f1': 0.5776566757493188, 'accuracy': 0.7690014903129657, 'auc': np.float64(0.8654781647318961), 'pr_auc': tensor(0.4020)}
----------------------------------------------------------------


In [5]:
import os
import numpy as np
from tqdm import tqdm

def validate_features(feature_dir):
    print(f"Scanning features in {feature_dir}...")
    files = [f for f in os.listdir(feature_dir) if f.endswith('.npz')]

    corrupt_files = []
    stats = {
        'total': len(files),
        'valid': 0,
        'corrupt': 0
    }

    for filename in tqdm(files):
        path = os.path.join(feature_dir, filename)
        try:
            data = np.load(path)
            if 'video_features' in data:
                feat = data['video_features']
            else:
                feat = data['arr_0']
            data.close()

            if np.isnan(feat).any():
                print(f"[NaN Found] {filename}")
                corrupt_files.append(filename)
                stats['corrupt'] += 1
            elif np.isinf(feat).any():
                print(f"[Inf Found] {filename}")
                corrupt_files.append(filename)
                stats['corrupt'] += 1
            else:
                stats['valid'] += 1

        except Exception as e:
            print(f"[Read Error] {filename}: {e}")
            corrupt_files.append(filename)
            stats['corrupt'] += 1

    print("\n--- Validation Summary ---")
    print(f"Total Files: {stats['total']}")
    print(f"Valid Files: {stats['valid']}")
    print(f"Corrupt Files: {stats['corrupt']}")

    return corrupt_files

# Run validation
feature_dir = "data/features/egovlp"
corrupt_files = validate_features(feature_dir)

if len(corrupt_files) > 0:
    print(f"\nIdentified {len(corrupt_files)} corrupt files. You should remove these or exclude them from the dataset.")
else:
    print("\nAll feature files are valid! The NaNs might be coming from Model Instability (Gradient Explosion).")

Scanning features in data/features/egovlp...


100%|██████████| 384/384 [00:00<00:00, 608.94it/s]


--- Validation Summary ---
Total Files: 384
Valid Files: 384
Corrupt Files: 0

All feature files are valid! The NaNs might be coming from Model Instability (Gradient Explosion).


In [11]:
from core.models.blocks import EncoderLayer, Encoder, MLP, fetch_input_dim
import torch.nn as nn
import torch

# --- STABILITY FIX: Redefine ErFormer with Pre-Norm ---
class StableErFormer(nn.Module):

    def __init__(self, config, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.config = config
        input_dimension = fetch_input_dim(config)

        # FIX: norm_first=True (Pre-Norm) stabilizes gradients significantly
        step_encoder_layer = EncoderLayer(
            d_model=input_dimension,
            dim_feedforward=2048,
            nhead=8,
            batch_first=True,
            norm_first=True  # <--- CRITICAL FIX FOR STABILITY
        )
        self.step_encoder = Encoder(step_encoder_layer, num_layers=1)

        decoder_input_dimension = fetch_input_dim(config, decoder=True)
        self.decoder = MLP(decoder_input_dimension, 512, 1)
        self.apply(self.init_weights)

    def forward(self, input_data):
        # Safety: clip inputs just in case
        input_data = torch.nan_to_num(input_data, nan=0.0, posinf=1.0, neginf=-1.0)

        encoded_output = self.step_encoder(input_data)
        _, dim = encoded_output.shape

        # (Logic from original ErFormer)
        video_output = encoded_output[:, :1024]
        if dim // 1024 == 1:
            encoded_output = video_output
        elif dim // 1024 == 2:
            audio_output = encoded_output[:, 1024:2048]
            encoded_output = 0.65 * video_output + 0.35 * audio_output
        # ... (simplified for video only case, keeping it generic)

        final_output = self.decoder(encoded_output)
        return final_output

    @staticmethod
    def init_weights(m):
        if isinstance(m, nn.Linear):
            torch.nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                torch.nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            torch.nn.init.constant_(m.bias, 0)
            torch.nn.init.constant_(m.weight, 1.0)

# Patch the fetch_model function to use our Stable Class
def fetch_stable_model(config):
    if config.variant == const.TRANSFORMER_VARIANT:
        model = StableErFormer(config)
    else:
        # Fallback to original for MLP
        from base import fetch_model as original_fetch
        model = original_fetch(config)

    model.to(config.device)
    return model

print("StableErFormer (Pre-Norm) class defined.")

StableErFormer (Pre-Norm) class defined.


In [12]:
# --- RUN STABLE TRAINING ---

# Reset Config for training
config.num_epochs = 15
config.lr = 1e-4  # We can likely use a higher LR now that it's stable
config.batch_size = 8

print(f"Starting Stable Training... (LR: {config.lr})")

try:
    # 1. Setup Data
    train_dataset = CaptainCookStepDataset(config, const.TRAIN, config.split)
    train_loader = torch.utils.data.DataLoader(train_dataset, collate_fn=collate_fn, **train_kwargs)

    val_dataset = CaptainCookStepDataset(config, const.VAL, config.split)
    val_loader = torch.utils.data.DataLoader(val_dataset, collate_fn=collate_fn, **test_kwargs)

    test_dataset = CaptainCookStepDataset(config, const.TEST, config.split)
    test_loader = torch.utils.data.DataLoader(test_dataset, collate_fn=collate_fn, **test_kwargs)

    # 2. Setup Model (Using Stable Version)
    model = fetch_stable_model(config)
    optimizer = optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    # 3. Training Loop (Simplified, no skipping needed hopefully)
    for epoch in range(config.num_epochs):
        model.train()
        train_loss = 0
        batches = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}")
        for data, target in pbar:
            data, target = data.to(config.device), target.to(config.device)
            optimizer.zero_grad()

            output = model(data)
            loss = criterion(output, target)

            if torch.isnan(loss):
                print("!!! CRITICAL: NaN Loss detected even with Pre-Norm. Reducing LR might be needed.")
                break

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            batches += 1
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        print(f"Epoch {epoch+1} Avg Loss: {train_loss/batches:.4f}")

        # Quick Val
        model.eval()
        val_loss = 0
        v_batches = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(config.device), target.to(config.device)
                output = model(data)
                val_loss += criterion(output, target).item()
                v_batches += 1
        print(f"Val Loss: {val_loss/v_batches:.4f}")

    # Test
    print("\nTesting Stable Model...")
    test_er_model(model, test_loader, criterion, config.device, "test")

except Exception as e:
    print(f"Error: {e}")

Starting Stable Training... (LR: 0.0001)
Loaded annotations...... 


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


Epoch 1/15:   0%|          | 3/3972 [00:00<00:42, 92.95it/s, loss=0.1615]

!!! CRITICAL: NaN Loss detected even with Pre-Norm. Reducing LR might be needed.
Epoch 1 Avg Loss: 0.6490


...

Epoch 2/15:   0%|          | 2/3972 [00:00<00:41, 95.26it/s, loss=0.2585]

!!! CRITICAL: NaN Loss detected even with Pre-Norm. Reducing LR might be needed.
Epoch 2 Avg Loss: 0.6739


Val Loss: nan


Epoch 3/15:   0%|          | 0/3972 [00:00<?, ?it/s]

!!! CRITICAL: NaN Loss detected even with Pre-Norm. Reducing LR might be needed.
Error: division by zero


In [17]:
from core.models.blocks import EncoderLayer, Encoder, MLP, fetch_input_dim
import torch.nn as nn
import torch

# --- STABILITY FIX V3: Tanh Bounding + Robust Shapes ---
class StableErFormer(nn.Module):

    def __init__(self, config, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.config = config
        input_dimension = fetch_input_dim(config)

        # 1. INPUT PROJECTION LAYER (Stabilizer)
        # Use Tanh instead of ReLU to bound inputs to [-1, 1]
        self.input_proj = nn.Sequential(
            nn.Linear(input_dimension, input_dimension),
            nn.LayerNorm(input_dimension),
            nn.Tanh(), # <--- V3 FIX: Bound inputs to prevent explosion
            nn.Dropout(0.1)
        )

        # 2. Pre-Norm Encoder
        step_encoder_layer = EncoderLayer(
            d_model=input_dimension,
            dim_feedforward=1024,
            nhead=4,
            batch_first=True,
            norm_first=True
        )
        self.step_encoder = Encoder(step_encoder_layer, num_layers=1)

        decoder_input_dimension = fetch_input_dim(config, decoder=True)
        self.decoder = MLP(decoder_input_dimension, 512, 1)
        self.apply(self.init_weights)

    def forward(self, input_data):
        # Safety: clip inputs
        input_data = torch.nan_to_num(input_data, nan=0.0, posinf=1.0, neginf=-1.0)

        # Project Input (Bounded to [-1, 1])
        x = self.input_proj(input_data)

        # Encode
        encoded_output = self.step_encoder(x)

        # ROBUST UNPACKING
        if encoded_output.dim() == 3:
            b, s, dim = encoded_output.shape
            encoded_output = encoded_output.reshape(-1, dim)
        else:
            _, dim = encoded_output.shape

        if dim >= 1024:
             video_output = encoded_output[:, :1024]
             if dim // 1024 == 1:
                 encoded_output = video_output

        final_output = self.decoder(encoded_output)
        return final_output

    @staticmethod
    def init_weights(m):
        if isinstance(m, nn.Linear):
            torch.nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                torch.nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            torch.nn.init.constant_(m.bias, 0)
            torch.nn.init.constant_(m.weight, 1.0)

def fetch_stable_model(config):
    model = StableErFormer(config)
    model.to(config.device)
    return model

print("StableErFormer V3 (Tanh Bounded) class defined.")

StableErFormer V3 (Tanh Bounded) class defined.


In [21]:
# --- RUN STABLE TRAINING (FINAL FIX: SKIP EMPTY BATCHES) ---

# Config: Switch back to Transformer now that we found the bug
config.variant = const.TRANSFORMER_VARIANT
config.num_epochs = 15
config.lr = 1e-5
config.batch_size = 1

print(f"Starting Final Stable Training (Transformer)... (LR: {config.lr})")

def fetch_robust_model(config):
    # Return the Stable Transformer V3
    model = StableErFormer(config)
    model.to(config.device)
    return model

try:
    # 1. Setup Data
    train_kwargs = {**cuda_kwargs, "shuffle": True, "batch_size": config.batch_size}
    test_kwargs = {**cuda_kwargs, "shuffle": False, "batch_size": config.test_batch_size}

    train_dataset = CaptainCookStepDataset(config, const.TRAIN, config.split)
    train_loader = torch.utils.data.DataLoader(train_dataset, collate_fn=collate_fn, **train_kwargs)

    val_dataset = CaptainCookStepDataset(config, const.VAL, config.split)
    val_loader = torch.utils.data.DataLoader(val_dataset, collate_fn=collate_fn, **test_kwargs)

    test_dataset = CaptainCookStepDataset(config, const.TEST, config.split)
    test_loader = torch.utils.data.DataLoader(test_dataset, collate_fn=collate_fn, **test_kwargs)

    # 2. Setup Model
    model = fetch_robust_model(config)
    optimizer = optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    # 3. Training Loop
    for epoch in range(config.num_epochs):
        model.train()
        train_loss = 0
        batches = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}")
        for data, target in pbar:
            data, target = data.to(config.device), target.to(config.device)

            # --- CRITICAL FIX: SKIP EMPTY BATCHES ---
            # If the tensor is empty (0 samples), skip it to avoid 0/0 NaN in loss
            if data.numel() == 0:
                continue

            # Ensure 3D input for Transformer: (1, Seq, Dim)
            # (Data comes as [Seq, Dim] from loader because batch_size=1)
            if data.dim() == 2:
                data = data.unsqueeze(0)

            optimizer.zero_grad()
            output = model(data)

            # Reshape output to match target
            if output.shape != target.shape:
                output = output.view(target.shape)

            loss = criterion(output, target)

            # Double check for NaNs (should be rare now)
            if torch.isnan(loss):
                print("!!! Warning: NaN Loss detected. Skipping batch.")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
            optimizer.step()

            train_loss += loss.item()
            batches += 1
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        if batches > 0:
            print(f"Epoch {epoch+1} Avg Loss: {train_loss/batches:.4f}")
        else:
             print("Epoch Failed: No valid batches.")

        # Validation
        model.eval()
        val_loss = 0
        v_batches = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(config.device), target.to(config.device)
                if data.numel() == 0: continue # Skip empty in Val too

                if data.dim() == 2: data = data.unsqueeze(0)
                output = model(data)
                if output.shape != target.shape: output = output.view(target.shape)
                l = criterion(output, target)
                if not torch.isnan(l):
                    val_loss += l.item()
                    v_batches += 1

        if v_batches > 0:
            print(f"Val Loss: {val_loss/v_batches:.4f}")
        else:
            print("Val Loss: NaN")

    # Test
    print("\nTesting...")
    def stable_test_wrapper(model, loader, crit, dev):
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in loader:
                data, target = data.to(dev), target.to(dev)
                if data.numel() == 0: continue

                if data.dim() == 2: data = data.unsqueeze(0)
                out = model(data)
                if out.shape != target.shape: out = out.view(target.shape)
                pred = (torch.sigmoid(out) > 0.5).float()
                correct += (pred == target).sum().item()
                total += target.numel()
        print(f"Test Accuracy: {correct/total:.4f}")

    stable_test_wrapper(model, test_loader, criterion, config.device)

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Starting Final Stable Training (Transformer)... (LR: 1e-05)
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


Epoch 1/15: 100%|██████████| 3972/3972 [00:22<00:00, 176.47it/s, loss=0.0055]


Epoch 1 Avg Loss: 1.5123
Val Loss: 1.9658


Epoch 2/15: 100%|██████████| 3972/3972 [00:21<00:00, 186.01it/s, loss=5.3498]


...

Epoch 13 Avg Loss: 1.9049
Val Loss: 2.3506


Epoch 14/15: 100%|██████████| 3972/3972 [00:21<00:00, 180.78it/s, loss=6.5567]


Epoch 14 Avg Loss: 1.9448
Val Loss: 2.2574


Epoch 15/15: 100%|██████████| 3972/3972 [00:22<00:00, 175.60it/s, loss=6.5822]


Epoch 15 Avg Loss: 1.9641
Val Loss: 2.2861

Testing...
Test Accuracy: 0.6350
